In [41]:
import numpy as np
from collections import Counter
from ArcMemory import ObjectState, PixelSet, ArcState
from helpers import determine_new_obj_props, mask_to_pixels, calculate_distance, get_direction_vector
import scipy.ndimage as ndi
from ArcDSL import _update_object_state, _update_arcstate_grid

In [3]:
def count_colours(
    colour_count: Counter,
) -> np.ndarray:
    """
    Use the colour counter to return a grid with the counts
    Return as a horizontal/vertical line (dependent on output shape) of pixels with a new row for each colour and the count as the length of the line
    """
    if not colour_count:
        return None
    if len(colour_count) == 1:
        return None
    n_colours = len(colour_count)
    max_count = max(colour_count.values())
    colour_count = Counter(
        dict(sorted(colour_count.items(), key=lambda x: x[1], reverse=False))
    )
    output_array = np.zeros((n_colours, max_count), dtype=int)
    for i, (colour, count) in enumerate(colour_count.items()):
        output_array[i, :count] = colour
    return output_array

In [4]:
count_colours(Counter({1: 3, 2: 5, 3: 2}))

array([[3, 3, 0, 0, 0],
       [1, 1, 1, 0, 0],
       [2, 2, 2, 2, 2]])

In [5]:
def fill_enclosed_area(
    obj: ObjectState,
    out_colour: int,
) -> ObjectState:
    """
    Fill any enclosed area of the object with the specified colour.
    Return a new ObjectState with the filled area (this will be a different object state than the input object as could be a different colour and area)
    """
    # Create a binary mask of the object
    binary_mask = obj.mask != 0
    filled_mask = ndi.binary_fill_holes(binary_mask)
    # Create a new array with the filled areas set to the specified colour
    filled_array = np.where(filled_mask, out_colour, obj.mask)
    new_pixels = mask_to_pixels(filled_array)
    if out_colour == obj.colour:
        combined_pixels = obj.cell_positions.union(new_pixels)
        return _update_object_state(obj, combined_pixels)
    else:
        return [ObjectState(
            label_id=obj.label_id + np.random.randint(1, 1000),
            colour=out_colour,
            grid_size=obj.grid_size,
            bounding_box=obj.bounding_box,
            centroid=obj.centroid,
            area=len(new_pixels),
            cell_positions=PixelSet(new_pixels),
        ),
        obj,
        ]

In [6]:
train ={            "input": [
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            ]}

obj = ArcState.from_array(np.array(train["input"])).objects[0]
new_obj = fill_enclosed_area(obj, out_colour=2)[0]
# Recreate grid with new cell positions
new_obj_cell_positions = new_obj.cell_positions
grid = np.zeros(obj.grid_size, dtype=int)
for r, c in new_obj_cell_positions:
    grid[r, c] = new_obj.colour
for r, c in obj.cell_positions:
    grid[r, c] = obj.colour  # Add original object colour

# Pretty print the grid
print("\n".join(" ".join(str(cell) for cell in row) for row in grid))

0 0 0 0 0 0 0 0 0 0 8 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 8 0 0 0 0 0 8 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 8 0 0 0 0 0 8 0 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 8 0 0 0 0 0 8 0 0 0 0 0 0
0 0 0 8 8 8 8 8 8 8 8 8 8 0 0 0 8 0 0 0 0 0 0
0 0 0 0 0 0 8 2 2 2 8 0 0 0 0 0 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 8 0 0 0 0 0 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 8 0 0 0 0 0 8 0 0 0 8 0 0
0 0 0 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 0
0 0 0 0 0 0 8 2 2 2 8 2 2 2 2 2 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 8 2 2 2 2 2 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 2 2 2 2 2 2 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 2 2 2 2 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 2 2 2 2 2 2 2 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 2 2 2 2 2 2 2 2 2 8 0 0 0 0 0 0
8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 0 0 0 8 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 0 0 0 8 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 0 0 0 8 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 0 0 8 8 8 8 8 8 8 8 8 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 8 0 0 8 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 8 0 0 8 

In [7]:
def _unfill_object(
    obj: ObjectState,
) -> ObjectState:
    """
    Unfill the object, only keeping the border of the object and setting the inside to background colour
    """
    cell_positions = obj.cell_positions
    new_cell_positions = set()
    # Get border cells only
    for cell in cell_positions:
        x, y = cell
        neighbors = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1),
        ]
        if any(neighbor not in cell_positions for neighbor in neighbors):
            new_cell_positions.add(cell)
    new_cell_positions = PixelSet(new_cell_positions)
    return _update_object_state(obj, new_cell_positions)


def unfill_all_objects(state: ArcState) -> ArcState:
    """
    Unfill all objects in the state, only keeping the borders of the objects and setting the inside to background colour
    """
    new_objects = []
    for obj in state.objects:
        new_obj = _unfill_object(obj)
        new_objects.append(new_obj)
    return ArcState(
        grid_state=state.grid_state,
        objects=tuple(new_objects),
    )


In [8]:
train = {             "input": [
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 8, 8, 8, 8, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            ],}

arc_state = ArcState.from_array(np.array(train["input"]))

new_state = unfill_all_objects(arc_state)

# Recreate grid with new cell positions
grid = np.zeros(arc_state.grid_state.dimensions, dtype=int)
for obj in new_state.objects:
    for r, c in obj.cell_positions:
        grid[r, c] = obj.colour

grid

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 8, 8, 8, 8, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
       [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0, 7, 0, 7, 0],
       [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0, 7, 0, 7, 0],
       [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 0, 7, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0,

In [9]:
def reflect_obj_relative_to_line_obj_keep_all_objects(
    state: ArcState,
) -> ArcState:
    """
    Reflect the object relative to the nearest line object in the specified direction
    """
    objects = state.objects
    line_objects = [obj for obj in objects if obj.has_line]
    # Get all other objects that are not line objects use opposite of intersection
    other_objects = [obj for obj in objects if obj not in line_objects]

    if not line_objects or not other_objects:
        return state

    print(f"Found {len(line_objects)} line objects and {len(other_objects)} other objects")

    # Filter line objects to only those that are within 1 cell of more than 2 other objects
    if len(line_objects) > 4:
        line_objects = [
            line
            for line in line_objects
            if sum(
                1
                for obj in other_objects
                if any(
                    np.linalg.norm(np.array(cell) - np.array(line_cell)) <= 1
                    for cell in obj.cell_positions
                    for line_cell in line.cell_positions
                )
            ) > 3
        ]
        print(f"Filtered line objects to {len(line_objects)} based on proximity to other objects")

    new_objects = []
    new_objects.extend(line_objects)
    new_objects.extend(other_objects)
    for obj in other_objects:
        # Find closest line
        closest_line = min(
            line_objects,
            key=lambda line: np.min(
                [
                    np.linalg.norm(np.array(cell) - np.array(line_cell))
                    for cell in obj.cell_positions
                    for line_cell in line.cell_positions
                ]
            ),
        )
        # Reflect the object relative to the closest line
        line_cells = list(closest_line.cell_positions)
        # Find most common row or column in the line cells to determine the mirror line
        rows, cols = zip(*line_cells)
        most_row, most_row_count = Counter(rows).most_common(1)[0]
        most_col, most_col_count = Counter(cols).most_common(1)[0]
        if most_row_count > most_col_count:
            # More rows than columns, mirror line is horizontal
            mirror_line = [(most_row, min(cols)), (most_row, max(cols))]
        else:
            # More columns than rows, mirror line is vertical
            mirror_line = [(min(rows), most_col), (max(rows), most_col)]
        new_cell_positions = set()
        for cell in obj.cell_positions:
            reflected = list(cell)
            # Axis 0 is vertical line (x coordinate is the same) and vice versa
            axis = 0 if mirror_line[0][0] == mirror_line[1][0] else 1
            reflected[axis] = mirror_line[0][axis] + (mirror_line[0][axis] - cell[axis])
            new_cell_positions.add(tuple(reflected))
        new_objects.append(_update_object_state(obj, PixelSet(new_cell_positions)))

    new_grid = np.zeros(state.grid_state.dimensions, dtype=int)
    try:
        for obj in new_objects:
            for r, c in obj.cell_positions:
                new_grid[r, c] = obj.colour
    except:
        return state
    return ArcState.from_array(new_grid)


In [10]:
train = {  "input": [
                [0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
                [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
                [0, 0, 0, 0, 6, 0, 0, 0, 4, 6, 4, 0, 0, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 4, 0, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 4, 4, 0, 6, 0, 4, 4, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 4, 0, 0, 6, 0, 0, 0, 0],
                [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
                [0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 4, 4, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 0, 0, 4, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
                [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
                [0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
                [0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
            ],}

arc_state = ArcState.from_array(np.array(train["input"]))
new_state = reflect_obj_relative_to_line_obj_keep_all_objects(arc_state)
new_state.grid_state.as_array

Found 1 line objects and 3 other objects


array([[0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 4, 0, 0, 6, 0, 0, 0, 0],
       [0, 0, 0, 0, 6, 0, 4, 4, 0, 6, 0, 4, 4, 0, 6, 0, 0, 0, 0],
       [0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 4, 0, 0, 6, 0, 0, 0, 0],
       [0, 0, 0, 0, 6, 0, 0, 0, 4, 6, 4, 0, 0, 0, 6, 0, 0, 0, 0],
       [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
       [0, 0, 0, 0, 6, 0, 0, 0, 4, 6, 4, 0, 0, 0, 6, 0, 0, 0, 0],
       [0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 4, 0, 0, 6, 0, 0, 0, 0],
       [0, 0, 0, 0, 6, 0, 4, 4, 0, 6, 0, 4, 4, 0, 6, 0, 0, 0, 0],
       [0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 4, 0, 0, 6, 0, 0, 0, 0],
       [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
       [0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
       [0, 0, 0, 0, 6, 0, 4, 4, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
       [0, 0, 0, 0, 6, 0, 0, 4, 0, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
       [0, 0, 0, 0, 6, 0, 0, 0, 4, 6, 0, 0, 0, 0, 6, 0, 0, 0, 0],
       [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
       [0,

In [231]:
train = {             "input": [
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            ],
            "output": [
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            ],}



In [232]:
def connect_objects_diag_specific(
    obj: ObjectState,
    direction_vector: tuple,
    out_colour: int,
    target_pos: tuple
):
    """
    Specific transformation where we create a joining line between this object to another pixel on the grid
    The line first moves diagonally in the direction of the other pixel, then moves horizontally or vertically to reach the other pixel when 1 row/col away from the other pixel
    Assume object is a single pixel
    """
    grid_shape = obj.grid_size
    new_cell_positions = set(obj.cell_positions)
    dx, dy = direction_vector
    start_point = list(obj.cell_positions)[0]
    x, y = start_point
    target_x, target_y = target_pos

    break_condition = None
    # Move diagonally until we are 1 row/col away from the target pixel
    while True:
        x += dx
        y += dy
        if 0 <= x < grid_shape[0] and 0 <= y < grid_shape[1]:
            new_cell_positions.add((x, y))
        else:
            break
        if (abs(x - target_x) < 2):
            break_condition = "row"
            break
        elif (abs(y - target_y) < 2):
            break_condition = "col"
            break

    # Move horizontally or vertically to reach the target pixel
    if break_condition == "row":
        step = 1 if target_y > y else -1
        for c in range(y, target_y + step, step):
            if 0 <= x < grid_shape[0] and 0 <= c < grid_shape[1]:
                new_cell_positions.add((x, c))
    elif break_condition == "col":
        step = 1 if target_x > x else -1
        for r in range(x, target_x + step, step):
            if 0 <= r < grid_shape[0] and 0 <= y < grid_shape[1]:
                new_cell_positions.add((r, y))
    else:
        return obj

    # Remove last entry in new_cell_positions regardless
    if break_condition == "row":
        new_cell_positions.remove((x, target_y))
    elif break_condition == "col":
        new_cell_positions.remove((target_x, y))
    
    
    if out_colour == obj.colour:
        return _update_object_state(obj, PixelSet(new_cell_positions), colour=out_colour)
    else:
        bbox, centroid, hu = determine_new_obj_props(new_cell_positions)
        return [ObjectState(
            label_id=obj.label_id + np.random.randint(1, 1000),
            colour=out_colour,
            grid_size=obj.grid_size,
            bounding_box=bbox,
            centroid=centroid,
            hu_moments=hu,
            area=len(new_cell_positions),
            cell_positions=PixelSet(new_cell_positions),
        ), obj]


In [233]:
# Check distance and direction between the two objects in array
arc_state = ArcState.from_array(np.array(train["input"]))
arc_state.objects[0].cell_positions, arc_state.objects[1].cell_positions

(frozenset({(1, 11)}), frozenset({(13, 3)}))

In [237]:
new_objs = connect_objects_diag_specific(
    ArcState.from_array(np.array(train["input"])).objects[0],
    direction_vector=(1, -1),
    out_colour=4,
    target_pos = (13, 3)
)
# create a new grid with the new object and the original object
new_array = np.array(train["input"])
for obj in new_objs:
    for r, c in obj.cell_positions:
        new_array[r, c] = obj.colour

new_array

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [259]:
train = {
    "input": [
    [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2], 
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
    [0, 0, 3, 0, 0, 0, 0, 3, 3, 0, 0, 0], 
    [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
    [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0], 
    [0, 0, 0, 0, 3, 3, 3, 0, 0, 0, 0, 0], 
    [0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0], 
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0], 
    [0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0], 
    [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2]]
}


In [266]:
def swap_colour_shrink_grid(input_array: np.ndarray, background_colour: int) -> np.ndarray:
    """
    Swap colours if two colours are present in the input array
    Shrink grid by removing the outer cells for rows and cols
    """
    colours = np.unique(input_array)
    colours = colours[colours != background_colour]
    print(f"Unique colours in input array: {colours}")
    if len(colours) == 2:
        mask_a = input_array == colours[0]
        mask_b = input_array == colours[1]
        new_array = np.select([mask_a, mask_b], [colours[1], colours[0]], default=input_array)
        return new_array[1:-1, 1:-1]
    return input_array[1:-1, 1:-1]

In [267]:
swapped_array = swap_colour_shrink_grid(np.array(train["input"]), background_colour=0)
swapped_array

Unique colours in input array: [2 3]


array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 2, 0, 0, 0, 0, 2, 2, 0, 0],
       [0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 2, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 2, 2, 2, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 2, 0, 0, 2, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 2, 0],
       [0, 0, 0, 0, 2, 2, 0, 0, 0, 0]])